In [ ]:
from fisherA2Z.fisher import Fisher

import pyccl as ccl
import matplotlib.pyplot as plt
import warnings
from fisherA2Z.fisher import Fisher, marginalize, SmailZ, Core, PhotozModel,plot_contours 
import glob
import numpy as np
import scipy
warnings.filterwarnings('ignore')

In [ ]:
cosmo = ccl.Cosmology(Omega_c=0.2666, 
                       Omega_b=0.049, 
                       h=0.6727, 
                       sigma8=0.831, 
                       n_s=0.9645, 
                       transfer_function='eisenstein_hu')


In [ ]:
def get_s8_error(fisher):
    oms8 = np.zeros(shape = (2,2))
    factor = 2.416
    oms8[0][0] = fisher[0][0]
    oms8[1][0] = fisher[1][0] * factor
    oms8[0][1] = fisher[1][0] * factor
    
    oms8[1][1] = fisher[1][0] * factor**2
    
    return np.sqrt(1/oms8[1][1])
    
def fom(fisher):
    fisher_inv = np.linalg.inv(fisher)
    det = np.linalg.det(fisher_inv)
    
    return 1/np.sqrt(det)

def fom_2d(fisher):
    fisher_inv = np.linalg.inv(fisher)
    det = fisher_inv[1][1]*fisher_inv[0][0] - fisher_inv[0][1]**2
    return 1/np.sqrt(det)

In [ ]:
neff = 1.0
f_3x2pt_y10 = Fisher(cosmo, probe='3x2pt', save_deriv='data/obj_deriv_3x2pt_y10_no_outliers.pkl', 
                  overwrite=False, neff_source = [1.]*5, outliers=[0.0]*5)
f_3x2pt_y10.process()
oliginal_cov = np.linalg.pinv(f_3x2pt_y10.invcov*1e-10)
om_sigma8 = marginalize(f_3x2pt_y10.fisher, 0, 1)

w0_wa = marginalize(f_3x2pt_y10.fisher, 3, 4)

s8_err = get_s8_error(om_sigma8)
om_sigma8_fom = fom_2d(om_sigma8)

w0_wa_fom = fom_2d(w0_wa)

# print(s8_err, om_sigma8_fom)
print("w0-wa FoM: ", w0_wa_fom)
inflated_cov = scipy.linalg.pinvh(f_3x2pt_y10.invcov, rtol = 1e-16)

In [ ]:
# # Assume `cov` is your covariance matrix
# eigvals = np.linalg.eigvalsh(f_3x2pt_y10.invcov)  # since cov is symmetric
# # eigvals = np.linalg.eigvalsh(oliginal_cov)  # since cov is symmetric

# # Sort in ascending order for easier interpretation
# eigvals_sorted = np.sort(eigvals)

# # Plot the eigenvalue distribution
# plt.figure()
# plt.semilogy(eigvals_sorted, marker='o', linestyle='-')
# plt.xlabel('Eigenvalue index')
# plt.ylabel('Eigenvalue (log scale)')
# plt.title('Eigenvalue spectrum of covariance matrix')
# plt.grid(True)
# plt.show()

In [ ]:
# np.max(eigvals_sorted)

In [ ]:
# neff = 0.81
# [1.0]*4+[1-(1-neff)*5]

# f_3x2pt_y10 = Fisher(cosmo, probe='3x2pt', save_deriv='data/obj_deriv_3x2pt_y10_no_outliers.pkl', 
#                   overwrite=False, neff_source = [1.0]*4+[1-(1-neff)*5], outliers=[0.0]*5)
# f_3x2pt_y10.process()
# om_sigma8 = marginalize(f_3x2pt_y10.fisher, 0, 1)
# s8_err = get_s8_error(om_sigma8)
# om_sigma8_fom = fom_2d(om_sigma8)
# print(s8_err, om_sigma8_fom)
# inflated_cov = scipy.linalg.pinvh(f_3x2pt_y10.invcov, rtol = 1e-16)

In [ ]:
# plt.imshow(inflated_cov/oliginal_cov, vmin = 0, vmax = 100)
# plt.colorbar()

In [ ]:
neff_list  = np.linspace(1.0,0.1,10)
s8_err_list = []
om_sigma8_fom_list = []
w0_wa_fom_list = []

for neff in neff_list:
    print(neff)

    f_3x2pt_y10 = Fisher(cosmo, probe='3x2pt', save_deriv='data/obj_deriv_3x2pt_y10_no_outliers.pkl', 
                          overwrite=False, neff_source = [neff]*5, outliers=[0.0]*5)

    prior = f_3x2pt_y10.priors.copy()
    prior_bias_by_1plusz = [0.008, 0.005, 0.003, 0.003, 0.003]
    prior_sigma_by_1plusz = [0.01, 0.007, 0.0034, 0.0025, 0.0021]
    prior_outlier = [0.0054, 0.0056, 0.053, 0.0052, 0.0039]

    for i in range(5):
        prior[f'zbias{i+1}'] = 1/prior_bias_by_1plusz[i]**2
        prior[f'zvariance{i+1}'] = 1/prior_sigma_by_1plusz[i]**2
        prior[f'zoutlier{i+1}'] = 1/prior_outlier[i]**2

    f_3x2pt_y10.override_priors(prior)

    f_3x2pt_y10.process()

    om_sigma8 = marginalize(f_3x2pt_y10.fisher, 0, 1)
    s8_err = get_s8_error(om_sigma8)
    om_sigma8_fom = fom_2d(om_sigma8)
    w0_wa = marginalize(f_3x2pt_y10.fisher, 3, 4)
    w0_wa_fom = fom_2d(w0_wa)
    
    # print(s8_err, om_sigma8_fom)
    print("w0-wa FoM: ", w0_wa_fom)
    
    s8_err_list.append(s8_err)
    om_sigma8_fom_list.append(om_sigma8_fom)
    w0_wa_fom_list.append(w0_wa_fom)



In [ ]:
neff_list_drop5 = [1.0, 0.95, 0.9, 0.85, 0.81]
s8_err_list_drop5 = []
om_sigma8_fom_list_drop5 = []
w0_wa_fom_list_drop5 = []

for neff in neff_list_drop5:
    print(neff)

    f_3x2pt_y10 = Fisher(cosmo, probe='3x2pt', save_deriv='data/obj_deriv_3x2pt_y10_no_outliers.pkl', 
                      overwrite=False, neff_source = [1.0]*4+[1-(1-neff)*5])

    prior = f_3x2pt_y10.priors.copy()
    prior_bias_by_1plusz = [0.008, 0.005, 0.003, 0.003, 0.003]
    prior_sigma_by_1plusz = [0.01, 0.007, 0.0034, 0.0025, 0.0021]
    prior_outlier = [0.0054, 0.0056, 0.053, 0.0052, 0.0039]

    for i in range(5):
        prior[f'zbias{i+1}'] = 1/prior_bias_by_1plusz[i]**2
        prior[f'zvariance{i+1}'] = 1/prior_sigma_by_1plusz[i]**2
        prior[f'zoutlier{i+1}'] = 1/prior_outlier[i]**2

    f_3x2pt_y10.override_priors(prior)

    f_3x2pt_y10.process()

    om_sigma8 = marginalize(f_3x2pt_y10.fisher, 0, 1)
    s8_err = get_s8_error(om_sigma8)
    om_sigma8_fom = fom_2d(om_sigma8)

    w0_wa = marginalize(f_3x2pt_y10.fisher, 3, 4)
    w0_wa_fom = fom_2d(w0_wa)
    
    # print(s8_err, om_sigma8_fom)
    print("w0-wa FoM: ", w0_wa_fom)
    
    s8_err_list_drop5.append(s8_err)
    om_sigma8_fom_list_drop5.append(om_sigma8_fom)
    w0_wa_fom_list_drop5.append(w0_wa_fom)



In [ ]:
neff_list_drop45 = [1.0, 0.95, 0.9, 0.85, 0.8, 0.75,0.7,0.65]
s8_err_list_drop45 = []
om_sigma8_fom_list_drop45 = []
w0_wa_fom_list_drop45 = []

for neff in neff_list_drop45:
    print(neff)

    f_3x2pt_y10 = Fisher(cosmo, probe='3x2pt', save_deriv='data/obj_deriv_3x2pt_y10_no_outliers.pkl', 
                      overwrite=False, neff_source = [1.0]*3+[1-(1-neff)*5/2, 1-(1-neff)*5/2])

    prior = f_3x2pt_y10.priors.copy()
    prior_bias_by_1plusz = [0.008, 0.005, 0.003, 0.003, 0.003]
    prior_sigma_by_1plusz = [0.01, 0.007, 0.0034, 0.0025, 0.0021]
    prior_outlier = [0.0054, 0.0056, 0.053, 0.0052, 0.0039]

    for i in range(5):
        prior[f'zbias{i+1}'] = 1/prior_bias_by_1plusz[i]**2
        prior[f'zvariance{i+1}'] = 1/prior_sigma_by_1plusz[i]**2
        prior[f'zoutlier{i+1}'] = 1/prior_outlier[i]**2

    f_3x2pt_y10.override_priors(prior)

    f_3x2pt_y10.process()

    om_sigma8 = marginalize(f_3x2pt_y10.fisher, 0, 1)
    s8_err = get_s8_error(om_sigma8)
    om_sigma8_fom = fom_2d(om_sigma8)
    w0_wa = marginalize(f_3x2pt_y10.fisher, 3, 4)
    w0_wa_fom = fom_2d(w0_wa)
    
    
    
    # print(s8_err, om_sigma8_fom)
    print("w0-wa FoM: ", w0_wa_fom)
    
    s8_err_list_drop45.append(s8_err)
    om_sigma8_fom_list_drop45.append(om_sigma8_fom)
    w0_wa_fom_list_drop45.append(w0_wa_fom)



In [ ]:
neff_list_drop345 = [1.0, 0.95, 0.9, 0.85, 0.8, 0.75,0.7,0.65,0.6, 0.55, 0.5, 0.45]
s8_err_list_drop345 = []
om_sigma8_fom_list_drop345 = []
w0_wa_fom_list_drop345 = []

for neff in neff_list_drop345:
    print(neff)

    f_3x2pt_y10 = Fisher(cosmo, probe='3x2pt', save_deriv='data/obj_deriv_3x2pt_y10_no_outliers.pkl', 
                      overwrite=False, neff_source = [1.0]*2+[1-(1-neff)*5/3]*3)

    prior = f_3x2pt_y10.priors.copy()
    prior_bias_by_1plusz = [0.008, 0.005, 0.003, 0.003, 0.003]
    prior_sigma_by_1plusz = [0.01, 0.007, 0.0034, 0.0025, 0.0021]
    prior_outlier = [0.0054, 0.0056, 0.053, 0.0052, 0.0039]

    for i in range(5):
        prior[f'zbias{i+1}'] = 1/prior_bias_by_1plusz[i]**2
        prior[f'zvariance{i+1}'] = 1/prior_sigma_by_1plusz[i]**2
        prior[f'zoutlier{i+1}'] = 1/prior_outlier[i]**2

    f_3x2pt_y10.override_priors(prior)

    f_3x2pt_y10.process()

    om_sigma8 = marginalize(f_3x2pt_y10.fisher, 0, 1)
    s8_err = get_s8_error(om_sigma8)
    om_sigma8_fom = fom_2d(om_sigma8)
    w0_wa = marginalize(f_3x2pt_y10.fisher, 3, 4)
    w0_wa_fom = fom_2d(w0_wa)
    
    
    
    # print(s8_err, om_sigma8_fom)
    print("w0-wa FoM: ", w0_wa_fom)
    
    s8_err_list_drop345.append(s8_err)
    om_sigma8_fom_list_drop345.append(om_sigma8_fom)
    w0_wa_fom_list_drop345.append(w0_wa_fom)



In [ ]:
plt.figure(figsize = (7,6))
plt.plot(neff_list,s8_err_list/s8_err_list[0], label = 'All bin equally' )
plt.plot(neff_list_drop5,s8_err_list_drop5/s8_err_list_drop5[0], label = 'Only fail in bin 5' )
plt.plot(neff_list_drop45,s8_err_list_drop45/s8_err_list_drop45[0], label = 'Only fail in bin 4 and 5' )
plt.plot(neff_list_drop345,s8_err_list_drop345/s8_err_list_drop345[0], label = 'Only fail in bin 3, 4 and 5' )

plt.title('3x2pt, LSST Y10')

plt.xlabel(r'$n_{eff}$ fraction')
plt.ylabel(r'$\Delta S_8 (n_{eff})/ \Delta S_8 (n_{eff,0})$')
plt.legend()
plt.xlim(1.05, 0.25)
plt.ylim(0.5, 3.5)

In [ ]:
plt.figure(figsize = (7,6))
plt.plot(neff_list,om_sigma8_fom_list/om_sigma8_fom_list[0], label = 'All bin equally' )
plt.plot(neff_list_drop5,om_sigma8_fom_list_drop5/om_sigma8_fom_list_drop5[0], label = 'Only fail in bin 5' )
plt.plot(neff_list_drop45,om_sigma8_fom_list_drop45/om_sigma8_fom_list_drop45[0], label = 'Only fail in bin 4 and 5' )
plt.plot(neff_list_drop345,om_sigma8_fom_list_drop345/om_sigma8_fom_list_drop345[0], label = 'Only fail in bin 3, 4 and 5' )

plt.title('3x2pt, LSST Y10')

plt.xlabel(r'$n_{eff}$ fraction')
plt.ylabel(r'$FoM (n_{eff})/ FoM (n_{eff,0})$')
plt.xlim(1.05, 0.05)
plt.ylim(-0.05, 1.05)

In [ ]:
plt.figure(figsize = (7,6))
plt.plot(neff_list,w0_wa_fom_list/w0_wa_fom_list[0], label = 'All bin equally' )
plt.plot(neff_list_drop5,w0_wa_fom_list_drop5/w0_wa_fom_list_drop5[0], label = 'Only fail in bin 5' )
plt.plot(neff_list_drop45,w0_wa_fom_list_drop45/w0_wa_fom_list_drop45[0], label = 'Only fail in bin 4 and 5' )
plt.plot(neff_list_drop345,w0_wa_fom_list_drop345/w0_wa_fom_list_drop345[0], label = 'Only fail in bin 3, 4 and 5' )

plt.title('3x2pt, LSST Y10')

plt.xlabel(r'$n_{eff}$ fraction')
plt.ylabel(r'$FoM (n_{eff})/ FoM (n_{eff,0})$')
plt.xlim(1.05, 0.05)
plt.ylim(-0.05, 1.05)

In [ ]:
print(neff_list, neff_list_drop5, neff_list_drop45, neff_list_drop345 )
# print(s8_err_list, s8_err_list_drop5, s8_err_list_drop45, s8_err_list_drop345)
# print(om_sigma8_fom_list, om_sigma8_fom_list_drop5, om_sigma8_fom_list_drop45,om_sigma8_fom_list_drop345)
print(w0_wa_fom_list, w0_wa_fom_list_drop5, w0_wa_fom_list_drop45,w0_wa_fom_list_drop345)

In [ ]:
f_3x2pt_y1 = Fisher(cosmo, probe='3x2pt', save_deriv='data/obj_deriv_3x2pt_y1_no_outliers.pkl', 
                  overwrite=False, neff_source = [neff]*5, outliers=[0.0]*5, y1 = True)
f_3x2pt_y1.process()

In [ ]:
prior = f_3x2pt_y1.priors.copy()
prior_bias_by_1plusz = [0.008, 0.005, 0.003, 0.003, 0.003]
prior_sigma_by_1plusz = [0.01, 0.007, 0.0034, 0.0025, 0.0021]
prior_outlier = [0.0054, 0.0056, 0.053, 0.0052, 0.0039]

for i in range(5):
    prior[f'zbias{i+1}'] = 1/prior_bias_by_1plusz[i]**2
    prior[f'zvariance{i+1}'] = 1/prior_sigma_by_1plusz[i]**2
    prior[f'zoutlier{i+1}'] = 1/prior_outlier[i]**2

In [ ]:
neff_list  = np.linspace(1.0,0.1,10)
s8_err_y1_list = []
om_sigma8_fom_y1_list = []
w0_wa_fom_y1_list = []

for neff in neff_list:
    print(neff)

    f_3x2pt_y1 = Fisher(cosmo, probe='3x2pt', save_deriv='data/obj_deriv_3x2pt_y1_no_outliers.pkl', 
                          overwrite=False, neff_source = [neff]*5, outliers=[0.0]*5, y1 = True)

    prior = f_3x2pt_y1.priors.copy()
    prior_bias_by_1plusz = [0.008, 0.005, 0.003, 0.003, 0.003]
    prior_sigma_by_1plusz = [0.01, 0.007, 0.0034, 0.0025, 0.0021]
    prior_outlier = [0.0054, 0.0056, 0.053, 0.0052, 0.0039]

    for i in range(5):
        prior[f'zbias{i+1}'] = 1/prior_bias_by_1plusz[i]**2
        prior[f'zvariance{i+1}'] = 1/prior_sigma_by_1plusz[i]**2
        prior[f'zoutlier{i+1}'] = 1/prior_outlier[i]**2

    f_3x2pt_y1.override_priors(prior)
    f_3x2pt_y1.process()

    om_sigma8 = marginalize(f_3x2pt_y1.fisher, 0, 1)
    s8_err = get_s8_error(om_sigma8)
    om_sigma8_fom = fom_2d(om_sigma8)
    
    w0_wa = marginalize(f_3x2pt_y1.fisher, 3, 4)
    w0_wa_fom = fom_2d(w0_wa)
    
    
    
    # print(s8_err, om_sigma8_fom)
    print("w0-wa FoM: ", w0_wa_fom)
    
    s8_err_y1_list.append(s8_err)
    om_sigma8_fom_y1_list.append(om_sigma8_fom)
    w0_wa_fom_y1_list.append(w0_wa_fom)



In [ ]:
neff_list_drop5 = [1.0, 0.95, 0.9, 0.85, 0.81]
s8_err_y1_list_drop5 = []
om_sigma8_fom_y1_list_drop5 = []
w0_wa_fom_y1_list_drop5 = []

for neff in neff_list_drop5:
    print(neff)

    f_3x2pt_y1 = Fisher(cosmo, probe='3x2pt', save_deriv='data/obj_deriv_3x2pt_y1_no_outliers.pkl', 
                      overwrite=False, neff_source = [1.0]*4+[1-(1-neff)*5], y1 = True)



    f_3x2pt_y1.override_priors(prior)

    f_3x2pt_y1.process()

    om_sigma8 = marginalize(f_3x2pt_y1.fisher, 0, 1)
    s8_err = get_s8_error(om_sigma8)
    om_sigma8_fom = fom_2d(om_sigma8)
    w0_wa = marginalize(f_3x2pt_y1.fisher, 3, 4)
    w0_wa_fom = fom_2d(w0_wa)
    
    
    
    # print(s8_err, om_sigma8_fom)
    print("w0-wa FoM: ", w0_wa_fom)
    
    s8_err_y1_list_drop5.append(s8_err)
    om_sigma8_fom_y1_list_drop5.append(om_sigma8_fom)
    w0_wa_fom_y1_list_drop5.append(w0_wa_fom)



In [ ]:
neff_list_drop45 = [1.0, 0.95, 0.9, 0.85, 0.8, 0.75,0.7,0.65]
s8_err_y1_list_drop45 = []
om_sigma8_fom_y1_list_drop45 = []
w0_wa_fom_y1_list_drop45 = []

for neff in neff_list_drop45:
    print(neff)

    f_3x2pt_y1 = Fisher(cosmo, probe='3x2pt', save_deriv='data/obj_deriv_3x2pt_y1_no_outliers.pkl', 
                      overwrite=False, neff_source = [1.0]*3+[1-(1-neff)*5/2, 1-(1-neff)*5/2], y1 = True)

    prior = f_3x2pt_y1.priors.copy()

    f_3x2pt_y1.override_priors(prior)

    f_3x2pt_y1.process()

    om_sigma8 = marginalize(f_3x2pt_y1.fisher, 0, 1)
    s8_err = get_s8_error(om_sigma8)
    om_sigma8_fom = fom_2d(om_sigma8)
    w0_wa = marginalize(f_3x2pt_y1.fisher, 3, 4)
    w0_wa_fom = fom_2d(w0_wa)
    
    
    
    # print(s8_err, om_sigma8_fom)
    print("w0-wa FoM: ", w0_wa_fom)
    
    s8_err_y1_list_drop45.append(s8_err)
    om_sigma8_fom_y1_list_drop45.append(om_sigma8_fom)
    w0_wa_fom_y1_list_drop45.append(w0_wa_fom)



In [ ]:
neff_list_drop345 = [1.0, 0.95, 0.9, 0.85, 0.8, 0.75,0.7,0.65,0.6, 0.55, 0.5, 0.45]
s8_err_y1_list_drop345 = []
om_sigma8_fom_y1_list_drop345 = []
w0_wa_fom_y1_list_drop345 = []

for neff in neff_list_drop345:
    print(neff)

    f_3x2pt_y1 = Fisher(cosmo, probe='3x2pt', save_deriv='data/obj_deriv_3x2pt_y1_no_outliers.pkl', 
                      overwrite=False, neff_source = [1.0]*2+[1-(1-neff)*5/3]*3, y1 = True)

    prior = f_3x2pt_y1.priors.copy()
    prior_bias_by_1plusz = [0.008, 0.005, 0.003, 0.003, 0.003]
    prior_sigma_by_1plusz = [0.01, 0.007, 0.0034, 0.0025, 0.0021]
    prior_outlier = [0.0054, 0.0056, 0.053, 0.0052, 0.0039]

    for i in range(5):
        prior[f'zbias{i+1}'] = 1/prior_bias_by_1plusz[i]**2
        prior[f'zvariance{i+1}'] = 1/prior_sigma_by_1plusz[i]**2
        prior[f'zoutlier{i+1}'] = 1/prior_outlier[i]**2

    f_3x2pt_y1.override_priors(prior)

    f_3x2pt_y1.process()

    om_sigma8 = marginalize(f_3x2pt_y1.fisher, 0, 1)
    s8_err = get_s8_error(om_sigma8)
    om_sigma8_fom = fom_2d(om_sigma8)
    w0_wa = marginalize(f_3x2pt_y1.fisher, 3, 4)
    w0_wa_fom = fom_2d(w0_wa)
    
    
    
    # print(s8_err, om_sigma8_fom)
    print("w0-wa FoM: ", w0_wa_fom)
    
    s8_err_y1_list_drop345.append(s8_err)
    om_sigma8_fom_y1_list_drop345.append(om_sigma8_fom)
    w0_wa_fom_y1_list_drop345.append(w0_wa_fom)



In [ ]:
plt.figure(figsize = (7,6))
plt.plot(neff_list,s8_err_y1_list/s8_err_y1_list[0], label = 'All bin equally' )
plt.plot(neff_list_drop5,s8_err_y1_list_drop5/s8_err_y1_list_drop5[0], label = 'Only fail in bin 5' )
plt.plot(neff_list_drop45,s8_err_y1_list_drop45/s8_err_y1_list_drop45[0], label = 'Only fail in bin 4 and 5' )
plt.plot(neff_list_drop345,s8_err_y1_list_drop345/s8_err_y1_list_drop345[0], label = 'Only fail in bin 3, 4 and 5' )

plt.title('3x2pt, LSST Y1')
plt.xlabel(r'$n_{eff}$ fraction')
plt.ylabel(r'$\Delta S_8 (n_{eff})/ \Delta S_8 (n_{eff,0})$')
plt.legend()
plt.xlim(1.05, 0.25)
plt.ylim(0.5, 3.5)

In [ ]:
plt.figure(figsize = (7,6))
plt.plot(neff_list,om_sigma8_fom_y1_list/om_sigma8_fom_y1_list[0], label = 'All bin equally' )
plt.plot(neff_list_drop5,om_sigma8_fom_y1_list_drop5/om_sigma8_fom_y1_list_drop5[0], label = 'Only fail in bin 5' )
plt.plot(neff_list_drop45,om_sigma8_fom_y1_list_drop45/om_sigma8_fom_y1_list_drop45[0], label = 'Only fail in bin 4 and 5' )
plt.plot(neff_list_drop345,om_sigma8_fom_y1_list_drop345/om_sigma8_fom_y1_list_drop345[0], label = 'Only fail in bin 3, 4 and 5' )

plt.title('3x2pt, LSST Y1')

plt.xlabel(r'$n_{eff}$ fraction')
plt.ylabel(r'$FoM (n_{eff})/ FoM (n_{eff,0})$')
plt.xlim(1.05, 0.05)
plt.ylim(-0.05, 1.05)
plt.legend()


In [ ]:
plt.figure(figsize = (7,6))
plt.plot(neff_list,w0_wa_fom_y1_list/w0_wa_fom_y1_list[0], label = 'All bin equally' )
plt.plot(neff_list_drop5,w0_wa_fom_y1_list_drop5/w0_wa_fom_y1_list_drop5[0], label = 'Only fail in bin 5' )
plt.plot(neff_list_drop45,w0_wa_fom_y1_list_drop45/w0_wa_fom_y1_list_drop45[0], label = 'Only fail in bin 4 and 5' )
plt.plot(neff_list_drop345,w0_wa_fom_y1_list_drop345/w0_wa_fom_y1_list_drop345[0], label = 'Only fail in bin 3, 4 and 5' )

plt.title('3x2pt, LSST Y1')

plt.xlabel(r'$n_{eff}$ fraction')
plt.ylabel(r'$FoM (n_{eff})/ FoM (n_{eff,0})$')
plt.xlim(1.05, 0.05)
plt.ylim(-0.05, 1.05)

In [ ]:
print(neff_list, neff_list_drop5, neff_list_drop45, neff_list_drop345 )
# print(s8_err_y1_list, s8_err_y1_list_drop5, s8_err_y1_list_drop45, s8_err_y1_list_drop345)
# print(om_sigma8_fom_y1_list, om_sigma8_fom_y1_list_drop5, om_sigma8_fom_y1_list_drop45,om_sigma8_fom_y1_list_drop345)
print(w0_wa_fom_y1_list, w0_wa_fom_y1_list_drop5, w0_wa_fom_y1_list_drop45,w0_wa_fom_y1_list_drop345)